# CS 443/543 — Assignment 1
## Build and Investigate a Tiny Transformer

**Due:** October 27, 2026, 11:59 PM  
**Weight:** **20 points** | **Individual assignment**  
**Computing:** Free Google Colab, including CPU-only sessions. No paid resources are required.

For this assignment, most of the code is provided. You will complete four important pieces, train one tiny language model, and run two guided experiments.

| Part | Points |
|---|---:|
| 1. Complete the Transformer | 8 |
| 2. Train and Observe | 4 |
| 3A. Break Causal Masking | 3 |
| 3B. Change Decoding | 3 |
| 4. Final Analysis | 2 |
| **Total** | **20** |


**Submission:** Submit this completed notebook (.ipynb) on Canvas with all outputs visible.


**AI use:** You may use AI tools for learning and debugging, but you must understand the code and results you submit. At the end, briefly state your AI use, if any, and one thing you verified yourself.


## 0. Setup — 0 points

Run this cell first. The notebook downloads Tiny Shakespeare. If the download fails, it uses a small built-in fallback corpus.


In [25]:
import math, random, time, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
random.seed(42)
torch.set_num_threads(min(torch.get_num_threads(), 2))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, '| Device:', device)

url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
try:
    text = urllib.request.urlopen(url, timeout=25).read().decode('utf-8')
    dataset_name = 'Tiny Shakespeare'
except Exception as exc:
    print('Download unavailable; using bundled fallback corpus.')
    lines = [
        'the little fox ran through the green forest.',
        'the quiet river ran beside the old stone bridge.',
        'in the morning the fox found a bright red apple.',
        'in the evening the fox returned to the forest.',
        'the river and the forest were calm in the morning.',
        'the river and the forest were calm in the morning.',
    ]
    rng = random.Random(111)
    text = ''.join(rng.choice(lines) for _ in range(6000))
    dataset_name = 'Bundled synthetic fallback'

chars = sorted(set(text))
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for c,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[int(i)] for i in ids)
ids = torch.tensor(encode(text), dtype=torch.long)
ntrain = int(0.9 * len(ids))
data = {'train': ids[:ntrain], 'val': ids[ntrain:]}
vocab_size = len(chars)

print('Dataset:', dataset_name)
print('Characters:', len(text), '| Vocabulary:', vocab_size)
print('Example:', repr(text[:100]))


PyTorch: 2.14.1+cu130 | Device: cuda
Dataset: Tiny Shakespeare
Characters: 1115394 | Vocabulary: 65
Example: 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'


## Model settings

We use a tiny model:

- context length = 32 characters
- embedding size = 64
- 4 attention heads
- 2 Transformer blocks
- batch size = 8

You do **not** need to change these settings.


In [ ]:
block_size = 32
batch_size = 8
n_embd = 64
n_head = 4
n_layer = 2
assert n_embd % n_head == 0

def get_batch(split, batch_size=batch_size):
    arr = data[split]
    offsets = torch.randint(0, len(arr) - block_size - 1, (batch_size,))
    x = torch.stack([arr[i:i + block_size] for i in offsets]).to(device)
    y = torch.stack([arr[i + 1:i + block_size + 1] for i in offsets]).to(device)
    return x, y

xb, yb = get_batch('train')
print('Input shape:', tuple(xb.shape), '| Target shape:', tuple(yb.shape))
print('Input example:', repr(decode(xb[0].cpu())))


Input shape: (8, 32) | Target shape: (8, 32)
Input example: 'T:\nO serpent heart, hid with a f'


# Part 1 — Complete the Transformer — **8 / 20 points**

Most of the implementation is provided. Complete only the four TODOs below.

### TODO 1 — Scaled attention scores — **1.5 points**
Compute the query-key similarity scores and divide by `sqrt(head_dim)`.

### TODO 2 — Causal mask + softmax — **1.5 points**
Prevent each position from attending to future positions, then convert scores into attention weights.


In [44]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model=n_embd, heads=n_head):
        super().__init__()
        self.heads = heads
        self.d_model = d_model
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, use_causal_mask=True):
        B, T, D = x.shape
        head_dim = D // self.heads

        # PROVIDED: create Q, K, and V and split into attention heads.
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.heads, head_dim).transpose(1, 2) # What am I looking for?
        k = k.view(B, T, self.heads, head_dim).transpose(1, 2) # What kind of information do I contain?
        v = v.view(B, T, self.heads, head_dim).transpose(1, 2) # # What information can I contribute?

        # TODO 1 (1.5 pts): compute scaled dot-product attention scores.
        # Expected shape: (B, heads, T, T)
        scores = q @ k.transpose(-2, -1) / (head_dim ** 0.5)

        print(f"The scores is:{scores}")

        # TODO 2 (1.5 pts): if use_causal_mask is True, mask future positions
        # with -inf, then apply softmax over the final dimension.
        if use_causal_mask:
            mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)

        # PROVIDED: combine values and concatenate heads.
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)




### TODO 3 — Residual updates — **1.0 point**
Complete the two residual updates in the Transformer block.

### TODO 4 — Token + position embeddings — **1.0 point**
Create the initial representation by adding token embeddings and position embeddings.

Everything else in the language model is provided.


In [45]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.attention = CausalSelfAttention()
        self.norm2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x, use_causal_mask=True):
        # TODO 3 (1.0 pt): two pre-norm residual updates.
        # Hint: x = x + something(...)
        x = x + self.attention(
            self.norm1(x),
            use_causal_mask=use_causal_mask,
        )
        x = x + self.mlp(self.norm2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([TransformerBlock() for _ in range(n_layer)])
        self.final_norm = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None, use_causal_mask=True):
        B, T = idx.shape
        assert T <= block_size
        positions = torch.arange(T, device=idx.device)

        # TODO 4 (1.0 pt): add token embeddings and position embeddings.
        x = self.token_embedding(idx) + self.position_embedding(positions)

        # PROVIDED: Transformer blocks, final normalization, and LM head.
        for block in self.blocks:
            x = block(x, use_causal_mask=use_causal_mask)
        x = self.final_norm(x)
        logits = self.lm_head(x)

        # PROVIDED: next-token cross-entropy loss.
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, vocab_size),
                targets.reshape(-1)
            )
        return logits, loss

model = TinyGPT().to(device)
print('Parameters:', sum(p.numel() for p in model.parameters()))


Parameters: 110529


### Three short explanations — **3.0 points**

**1. Causal masking — 1.0 point**  
Why must a next-token language model be prevented from seeing future tokens?

_Write 2–3 sentences here._

**2. Attention vs. MLP — 1.0 point**  
What is the main difference between what attention does and what the MLP does?

_Write 2–3 sentences here._

**3. Position — 1.0 point**  
Why does the model need positional information?

_Write 2–3 sentences here._


### Check your implementation — required, no separate points

Run the cell below. Both checks should pass before you continue.


In [46]:
def run_checks(model):
    test_x, test_y = get_batch('train')
    logits, loss = model(test_x, targets=test_y)
    assert logits.shape == (batch_size, block_size, vocab_size)
    assert loss is not None and torch.isfinite(loss)
    print('✓ Shape and loss check passed. Initial loss:', round(float(loss), 3))

    model.eval()
    probe = test_x[:2].clone()
    changed = probe.clone()
    changed[:, -1] = (changed[:, -1] + 1) % vocab_size
    with torch.no_grad():
        a, _ = model(probe, use_causal_mask=True)
        b, _ = model(changed, use_causal_mask=True)
    assert torch.allclose(a[:, :-1], b[:, :-1], atol=1e-5)
    print('✓ Causal future-independence check passed.')
    model.train()

# Uncomment after completing TODOs:
run_checks(model)


The scores is:tensor([[[[-5.8727e-02,  8.6482e-02,  1.4498e-01,  ...,  1.3798e-02,
            2.3912e-01,  3.5328e-01],
          [ 2.4588e-01, -7.4893e-01, -4.1774e-01,  ...,  3.6246e-01,
            4.4843e-01,  3.3623e-01],
          [-1.2148e-01, -3.2797e-01, -6.1295e-02,  ...,  3.6764e-01,
            2.0122e-01,  1.6458e-01],
          ...,
          [ 1.7952e-01,  3.6258e-01, -5.9664e-02,  ...,  1.6597e-01,
           -3.2171e-01, -1.2808e-01],
          [ 1.8907e-01, -1.6462e-01, -2.4265e-01,  ...,  2.6467e-01,
            4.5202e-01, -1.1486e-01],
          [ 1.4146e-01,  6.3609e-01,  1.9403e-01,  ...,  2.1629e-01,
            5.8656e-02, -1.9536e-01]],

         [[ 2.5995e-02,  5.4669e-01,  3.9921e-01,  ..., -5.0581e-01,
           -3.1906e-01,  5.1616e-02],
          [-8.1624e-02, -1.0669e-01, -5.0078e-01,  ...,  3.7290e-01,
           -1.4180e-01, -1.8879e-01],
          [-3.9732e-01,  3.3016e-01, -1.0823e-01,  ..., -3.1964e-02,
           -3.6896e-01, -2.2903e-01],
      

/tmp/ipykernel_1173799/466621017.py:6: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  print('✓ Shape and loss check passed. Initial loss:', round(float(loss), 3))


# Part 2 — Train and Observe — **4 / 20 points**

You do not need to write a training loop. It is provided.

### Before training — **0.5 point**
What do you expect text from an untrained model to look like?

_Write one sentence here before running the next cells._


In [47]:
@torch.no_grad()
def generate(model, start_ids, new_tokens=120, temperature=1.0, greedy=False):
    model.eval()
    idx = torch.tensor(start_ids, device=device, dtype=torch.long).unsqueeze(0)
    for _ in range(new_tokens):
        window = idx[:, -block_size:]
        logits, _ = model(window)
        next_logits = logits[:, -1, :]
        if greedy:
            choice = next_logits.argmax(dim=-1, keepdim=True)
        else:
            probs = F.softmax(next_logits / temperature, dim=-1)
            choice = torch.multinomial(probs, 1)
        idx = torch.cat([idx, choice], dim=1)
    return decode(idx[0].cpu().tolist())

# For your final run, create ONE fresh model here and use this same model below.
# Uncomment after Part 1 works:
# model = TinyGPT().to(device)
# prefix = text[:15]
# start_ids = encode(prefix)
# print('Prefix:', repr(prefix))
# before_text = generate(model, start_ids, greedy=True)
# print('BEFORE TRAINING:', before_text)


In [48]:
@torch.no_grad()
def estimate_loss(model, batches=6):
    model.eval()
    result = {}
    for split in ('train', 'val'):
        losses = []
        for _ in range(batches):
            xx, yy = get_batch(split)
            _, loss = model(xx, yy)
            losses.append(float(loss))
        result[split] = sum(losses) / len(losses)
    model.train()
    return result

# PROVIDED TRAINING LOOP
# First try steps=30 as a smoke test. For submission, restart with a fresh model
# and use steps=400.
#
# optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
# steps = 400
# history = []
# start_time = time.monotonic()
#
# for step in range(steps + 1):
#     if step % 100 == 0 or step == steps:
#         stats = estimate_loss(model)
#         history.append({'step': step, **stats})
#         print('step', step, 'train', round(stats['train'], 3),
#               'val', round(stats['val'], 3))
#     if step == steps:
#         break
#     xb, yb = get_batch('train')
#     _, loss = model(xb, yb)
#     optimizer.zero_grad(set_to_none=True)
#     loss.backward()
#     optimizer.step()
#
# elapsed = time.monotonic() - start_time
# print('Training time (seconds):', round(elapsed, 1))
#
# plt.plot([r['step'] for r in history], [r['train'] for r in history], label='train')
# plt.plot([r['step'] for r in history], [r['val'] for r in history], label='validation')
# plt.xlabel('Training step')
# plt.ylabel('Cross-entropy loss')
# plt.legend()
# plt.show()
#
# after_text = generate(model, start_ids, greedy=True)
# print('AFTER TRAINING:', after_text)


### What to provide for Part 2

**Training evidence — 1.5 points**
- **0.5:** visible training + validation loss plot
- **0.5:** generated text before and after training
- **0.5:** parameter count and approximate training time

**Interpretation — 2.0 points**

**1. Evidence of learning — 1.0 point**  
Give **two specific observations** showing that the model learned something.

_Write here._

**2. Limitation — 1.0 point**  
Give **one specific limitation** visible in your trained model or generated text.

_Write here._


# Part 3A — Break Causal Masking — **3 / 20 points**

The experiment code is provided. You are studying what the causal mask prevents.

### Prediction — **0.5 point**
Before running the code:

> If I change a future token, I predict an earlier prediction **will / will not** change when the causal mask is ON, and **will / will not** change when it is OFF.

_Write your prediction here._


In [49]:
# PROVIDED EXPERIMENT — no retraining required.
# Uncomment after training.
#
# model.eval()
# a, _ = get_batch('val', batch_size=2)
# b = a.clone()
# b[:, -1] = (b[:, -1] + 1) % vocab_size
#
# print('Original ending:', repr(decode(a[0, -8:].cpu())))
# print('Changed ending :', repr(decode(b[0, -8:].cpu())))
#
# with torch.no_grad():
#     masked_a, _ = model(a, use_causal_mask=True)
#     masked_b, _ = model(b, use_causal_mask=True)
#     open_a, _ = model(a, use_causal_mask=False)
#     open_b, _ = model(b, use_causal_mask=False)
#
# masked_delta = (masked_a[:, :-1] - masked_b[:, :-1]).abs().max().item()
# unmasked_delta = (open_a[:, :-1] - open_b[:, :-1]).abs().max().item()
#
# print('Earlier-logit change, mask ON :', masked_delta)
# print('Earlier-logit change, mask OFF:', unmasked_delta)


### Explain Experiment A — **2.5 points**

**Result — 0.5 point**  
Record the two reported logit changes (mask ON and mask OFF).

_Write here._

**Why did this happen? — 1.0 point**  
Explain the difference in plain language.

_Write 2–4 sentences here._

**Why would training without the mask be “cheating”? — 1.0 point**

_Write 2–4 sentences here._


# Part 3B — Change Decoding — **3 / 20 points**

Use the **same trained model and same prefix**. Only the decoding strategy changes.

### Prediction — **0.5 point**
What do you expect to happen as temperature increases?

_Write one sentence here._


In [50]:
# PROVIDED EXPERIMENT
# Uncomment after training.
#
# print('Prefix:', repr(prefix))
#
# greedy_output = generate(model, start_ids, greedy=True)
# print('GREEDY:', greedy_output)
#
# torch.manual_seed(2026)
# temp07_output = generate(model, start_ids, temperature=0.7)
# print('TEMPERATURE 0.7:', temp07_output)
#
# torch.manual_seed(2026)
# temp13_output = generate(model, start_ids, temperature=1.3)
# print('TEMPERATURE 1.3:', temp13_output)


### Explain Experiment B — **2.5 points**

**Outputs — 0.5 point**  
Keep all three generated outputs visible.

**What changed? — 1.0 point**  
Describe one or two specific differences you observe across the outputs.

_Write here._

**Did the model itself change? — 0.5 point**  
Explain briefly.

_Write here._

**Can these three samples prove that one decoding strategy is “best”? — 0.5 point**

_Write here._


# Part 4 — Final Analysis — **2 / 20 points**

Complete **only the version for your course number**.

### CS 443
What was the most important thing you learned from building or experimenting with this model? Use **one specific result from your notebook** as evidence.

_Write about one short paragraph here._

### CS 543
State **one small hypothesis** about your trained model and test it without retraining. For example, change the amount of available context, remove positional information at evaluation, or change the sampling seed.

Report briefly:

**Hypothesis → What you changed → What you measured → What happened → What you conclude**

_Write your response here._


## AI Use Statement — required

**AI tools used (if any):**  
_Write here._

**What did you use them for?**  
_Write here._

**Give one example of something you verified yourself.**  
_Write here._

---

## Submission checklist

- [ ] Four TODOs completed
- [ ] Both implementation checks pass
- [ ] 400-step final training run completed
- [ ] Loss plot visible
- [ ] Before/after generations visible
- [ ] Experiment A prediction, result, and explanation completed
- [ ] Experiment B prediction, outputs, and explanation completed
- [ ] CS 443 or CS 543 final analysis completed
- [ ] AI Use Statement completed

**Total: 20 points**
